# Observable RAG Planning: Steps, Evidence, and Completion

| Field | Value |
|---|---|
| Stage | Autonomous RAG patterns |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Planning should expose tasks and evidence requirements, not depend on hidden chain-of-thought.

## 30-Second Summary

This notebook turns a two-part index-deletion question into an explicit plan: retrieve deletion policy, retrieve recovery policy, verify both sources, and synthesize. The plan is state a reviewer can inspect and test.

## Why This Matters

Multi-part questions need more than one source. Hidden reasoning cannot prove which requirements were covered or why the workflow stopped.

## Scope

| Covers | Does not cover |
|---|---|
| Explicit task plan, evidence slots, completion check, cited synthesis | Hidden reasoning capture, open-ended web planning, hosted LLM |


## Mental Model

```text
question -> task list -> retrieve each evidence slot -> coverage gate -> cited answer
```


In [1]:
DOCUMENTS = {
    "deletion": "Deleting an index requires owner approval and a recorded change ticket.",
    "recovery": "A deleted index can be rebuilt from the latest validated snapshot.",
}
question = "Can I delete an index, and how would I recover it?"
required_slots = {"deletion_policy", "recovery_policy"}


## How It Works

A planner emits named tasks with expected source IDs. Executors fill evidence slots; a coverage gate blocks synthesis until every required slot is present. The task list is concise operational state, not a transcript of private reasoning.


## Baseline

A single retrieval returns only the deletion policy, covering half the requested evidence.


In [2]:
baseline_evidence = {"deletion_policy": {"source": "deletion", "text": DOCUMENTS["deletion"]}}
baseline_coverage = len(set(baseline_evidence) & required_slots) / len(required_slots)
baseline_coverage


0.5

## Technique Implementation

The plan has one retrieval task per evidence slot followed by verification and synthesis. Each task declares its output rather than an unbounded natural-language intention.


In [3]:
plan = [
    {"task": "retrieve", "slot": "deletion_policy", "source": "deletion"},
    {"task": "retrieve", "slot": "recovery_policy", "source": "recovery"},
    {"task": "verify_coverage", "requires": sorted(required_slots)},
    {"task": "synthesize", "requires": sorted(required_slots)},
]

evidence = {
    task["slot"]: {"source": task["source"], "text": DOCUMENTS[task["source"]]}
    for task in plan if task["task"] == "retrieve"
}
plan, evidence


([{'task': 'retrieve', 'slot': 'deletion_policy', 'source': 'deletion'},
  {'task': 'retrieve', 'slot': 'recovery_policy', 'source': 'recovery'},
  {'task': 'verify_coverage',
   'requires': ['deletion_policy', 'recovery_policy']},
  {'task': 'synthesize', 'requires': ['deletion_policy', 'recovery_policy']}],
 {'deletion_policy': {'source': 'deletion',
   'text': 'Deleting an index requires owner approval and a recorded change ticket.'},
  'recovery_policy': {'source': 'recovery',
   'text': 'A deleted index can be rebuilt from the latest validated snapshot.'}})

## Controlled Experiment

We compare evidence coverage, assert that synthesis dependencies are explicit, and create an answer only after the coverage gate passes.


In [4]:
coverage = len(set(evidence) & required_slots) / len(required_slots)
can_synthesize = required_slots <= set(evidence)
answer = None
if can_synthesize:
    answer = (
        f"{evidence['deletion_policy']['text']} [deletion] "
        f"{evidence['recovery_policy']['text']} [recovery]"
    )
results = {"baseline_coverage": baseline_coverage, "planned_coverage": coverage, "can_synthesize": can_synthesize, "answer": answer}
results


{'baseline_coverage': 0.5,
 'planned_coverage': 1.0,
 'can_synthesize': True,
 'answer': 'Deleting an index requires owner approval and a recorded change ticket. [deletion] A deleted index can be rebuilt from the latest validated snapshot. [recovery]'}

## Evaluation

The one-shot baseline covers **1/2** evidence slots. The explicit plan covers **2/2**, declares synthesis dependencies, and cites both sources. This demonstrates task completeness, not general planner intelligence.


In [5]:
assert results["baseline_coverage"] == 0.5 and results["planned_coverage"] == 1.0
assert results["can_synthesize"] and all(f"[{source}]" in answer for source in DOCUMENTS)
assert plan[-1]["requires"] == sorted(required_slots)
print("Observable planning checks passed.")


Observable planning checks passed.


## Decision Guide

| Question shape | Approach |
|---|---|
| Single fact | Direct RAG |
| Known multi-part requirements | Explicit task plan |
| Dependent steps | DAG/sequential workflow |
| Missing required evidence | Stop/abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Missing clause | Plan omitted slot | Compare plan to question requirements |
| Synthesis starts early | No coverage gate | Declare dependencies |
| Plan grows forever | No task budget | Cap tasks/depth |
| Plan text leaks reasoning | Free-form scratchpad stored | Keep concise task/action state |


## Production Notes

### Observability
Trace task IDs, dependencies, evidence slots, source IDs, status, latency, and terminal reason.

### Safety and Guardrails
Plans cannot expand authorization or invent new tools.

### Latency and Cost
Parallelize independent tasks and cap total tasks/retries.


## Practice

Add an approval-owner clause and update the required slots before adding retrieval code.

## Recall

Toggle - Recall: What makes a plan observable?
Named tasks, dependencies, evidence slots, and terminal conditions.

Toggle - Recall: Why avoid hidden reasoning logs?
Operational state is enough to test control flow without collecting private rationale.

## Sources

- [LangGraph workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the explicit coverage contract | Add task failures and dynamic dependency validation |
